# Task 4 output profile

Checks one output file: uniqueness, empty values, prices and sets. Set `PATH` to the file to check.

In [50]:
import csv, json
from collections import Counter

PATH = "output/task4-us.csv"

with open(PATH, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(f"{PATH}: {len(rows)} rows, {len({r['product_id'] for r in rows})} products")

output/task4-us.csv: 330 rows, 96 products


In [51]:
# Uniqueness: each row is one variant.
keys = {
    "variant_id": lambda r: r["variant_id"],
    "url": lambda r: r["url"],
    "product + variant name": lambda r: (r["product_name"], r["variant_name"]),
}
for label, key in keys.items():
    counts = Counter(key(r) for r in rows)
    repeated = {k: c for k, c in counts.items() if c > 1}
    print(f"{label:24} {len(counts):>4} distinct  -> {'all unique' if not repeated else f'{len(repeated)} repeated'}")
    for k, c in list(repeated.items())[:10]:
        print(f"   x{c}: {k}")

variant_id                330 distinct  -> all unique
url                       330 distinct  -> all unique
product + variant name    330 distinct  -> all unique


In [52]:
# Empty values per column.
print({col: n for col in rows[0] if (n := sum(not r[col] for r in rows))})

{'variant_name': 57, 'image': 4, 'description': 10, 'price': 4, 'discounted_price': 302, 'flexible_discount_percent': 326, 'set_variant_ids': 305}


In [53]:
# Every row should have a price, or (build-your-own sets) a discount percentage instead.
percent_only = [r for r in rows if not r["price"] and r["flexible_discount_percent"]]
neither = [r for r in rows if not r["price"] and not r["flexible_discount_percent"]]
print(f"{sum(bool(r['price']) for r in rows)} with a price, {len(percent_only)} with a discount percentage only, {len(neither)} with neither")
for r in percent_only:
    print(f"   percentage only: {r['product_name']} ({r['flexible_discount_percent']}%)")
for r in neither:
    print(f"   NEITHER: {r['product_name']} {r['variant_name']} -> {r['url']}")

326 with a price, 4 with a discount percentage only, 0 with neither
   percentage only: Fragrance Duo (15.0%)
   percentage only: Travel Spray + Balm (10.0%)
   percentage only: Fragrance Two Ways (15.0%)
   percentage only: Fragrance Trio (21.0%)


In [54]:
# Sets: every variant ID in set_variant_ids should have its own row (variant_id) in this file.
known = {int(r["variant_id"]) for r in rows}
sets = [r for r in rows if r["set_variant_ids"]]
print(f"{len(sets)} sets")
for r in sets:
    slots = json.loads(r["set_variant_ids"])
    missing = sorted({i for slot in slots for i in slot} - known)
    empty = sum(not slot for slot in slots)
    note = (f" | {len(missing)} IDs without a row: {missing}" if missing else "") + (f" | {empty} empty slot(s)" if empty else "")
    print(f"   {r['product_name']:28} slots={[len(s) for s in slots]}{note}")

25 sets
   Fragrance Duo                slots=[5, 5]
   In A New York Minute Set     slots=[12, 10, 7, 9]
   Travel Spray + Balm          slots=[5, 12]
   Fragrance Two Ways           slots=[5, 5]
   The Glossier Icons           slots=[1, 12, 7, 1]
   Fragrance Trio               slots=[5, 5, 5]
   The Skincare Icons           slots=[1, 1, 1]
   The Face Kit                 slots=[12, 32]
   Boy Brow Duo                 slots=[7, 7]
   Balm Dotcom Trio             slots=[12, 12, 12]
   The No-Makeup Makeup Kit     slots=[7, 2, 32]
   Deodorant                    slots=[3, 1]
   Ultralip Duo                 slots=[9, 9]
   Balm + Line Duo              slots=[9, 12]
   You Look Good Pet Set        slots=[3, 1, 1, 1]
   The Curbside Set             slots=[1, 1]
   Lash Slick Duo               slots=[2, 2]
   Cloud Paint Duo              slots=[15, 15]
   Full Body Routine            slots=[3, 2, 2]
   The Cheek Kit                slots=[1, 15]
   The Glossier You Routine     slots=[1, 1]


In [55]:
# Images: each row should hold a one-item JSON array with an https URL. Expected outliers: the 4 build-your-own sets (no gallery in the HTML).
# Also flags one image shared by different products (a wrong-photo pick would show up here).
if "image" not in rows[0]:
    print("No image column in this file, re-run the spider")
else:
    outliers = []
    for r in rows:
        try:
            images = json.loads(r["image"]) if r["image"] else []
        except json.JSONDecodeError:
            images = None
        if not images:
            problem = "empty" if images == [] else "not a JSON array"
        elif len(images) != 1:
            problem = f"{len(images)} images"
        elif not images[0].startswith("https://"):
            problem = f"bad URL {images[0]}"
        else:
            continue
        outliers.append((r, problem))
    print(f"{len(outliers)} image outliers")
    for r, problem in outliers:
        print(f"   {r['variant_id']}  {r['product_name']:28} {r['variant_name']:20} flexible={r['is_flexible_set']:5} -> {problem}")

    products_by_image = {}
    for r in rows:
        if r["image"]:
            products_by_image.setdefault(r["image"], {})[r["product_id"]] = r["product_name"]
    shared = {img: p for img, p in products_by_image.items() if len(p) > 1}
    print(f"{len(shared)} images shared by different products")
    for img, products in shared.items():
        print(f"   {img}")
        for r in rows:
            if r["image"] == img:
                print(f"      {r['variant_id']}  {r['product_name']} {r['variant_name']}")

4 image outliers
   48145894605045  Fragrance Duo                                     flexible=True  -> empty
   48145895719157  Travel Spray + Balm                               flexible=True  -> empty
   48145895489781  Fragrance Two Ways                                flexible=True  -> empty
   48145911644405  Fragrance Trio                                    flexible=True  -> empty
0 images shared by different products


In [56]:
# Products with the most variants, and how many variants are out of stock.
print(f"{sum(r['in_stock'] == 'False' for r in rows)} variants out of stock")
print("most variants:", Counter(r["product_name"] for r in rows).most_common(5))

3 variants out of stock
most variants: [('Stretch Balm Concealer', 32), ('Stretch Fluid Foundation', 32), ('Cloud Paint', 15), ('Balm Dotcom', 12), ('Perfecting Skin Tint', 12)]
